# APIM ❤️ Foundry IQ

## Foundry IQ with Microsoft Agent Framework via APIM lab

![flow](../../images/foundry-iq-agentfw.gif)

This lab integrates the **Microsoft Agent Framework** with a **Foundry IQ Knowledge Base** using the `AzureAISearchContextProvider`, with **Azure API Management** serving as the gateway for embedding traffic. The Agent Framework's context provider handles retrieval directly from Azure AI Search, supporting both **semantic** (fast hybrid search) and **agentic** (intelligent multi-hop) retrieval modes.

### Key concepts

- **Foundry IQ Knowledge Base** — Azure AI Search's managed Agentic Retrieval pipeline
- **AzureAISearchContextProvider** — Agent Framework context provider that plugs Foundry IQ into any agent
- **APIM as AI Gateway** — All OpenAI embedding traffic flows through APIM with token metrics and managed identity auth
- **Two retrieval modes** — Semantic (fast hybrid search) and Agentic (Foundry IQ multi-hop reasoning)

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

In [ ]:
%pip install 'azure-search-documents==11.7.0b2' 'azure-identity' 'openai' 'agent-framework-azure-ai-search' --pre 'agent-framework==1.0.0rc5' --pre

<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management)
- Adjust the models and versions according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models)

In [ ]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "swedencentral"

# AI Services configuration
aiservices_config = [{"name": "foundry", "location": "swedencentral"}]

embedding_model = "text-embedding-3-large"
agent_model = "gpt-4.1-mini"

# Models configuration - GPT-4.1-mini for agent + text-embedding-3-large for vectorizer
models_config = [
    {"name": agent_model, "publisher": "OpenAI", "version": "2025-04-14", "sku": "GlobalStandard", "capacity": 50},
    {"name": embedding_model, "publisher": "OpenAI", "version": "1", "sku": "GlobalStandard", "capacity": 120}
]

# APIM configuration
apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

# API configuration
ai_search_api_path = "ai-search"
inference_api_path = "inference"
inference_api_type = "AzureOpenAI"
foundry_project_name = deployment_name

# Knowledge Base configuration
index_name = "foundry-iq-idx"
knowledge_source_name = "foundry-iq-ks"
knowledge_base_name = "foundry-iq-kb"

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all the resources that will be deployed in the specified resource group. This includes:

- **Azure API Management** — Gateway for all inference traffic with managed identity auth
- **Azure AI Foundry** — Cognitive Services account + Project for model deployments
- **Azure AI Search** — Search service for Knowledge Base with semantic ranking
- **Log Analytics + Application Insights** — Monitoring and diagnostics

Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

In [ ]:
# Create the resource group if it doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name },
        "searchServiceLocation": { "value": resource_group_location },
        "aiSearchAPIPath": { "value": ai_search_api_path }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the gateway URL, subscription key, search endpoint, and Foundry project endpoint from the Bicep deployment.

In [ ]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}",
    f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get("key")
    foundry_project_endpoint = utils.get_deployment_output(output, 'foundryProjectEndpoint', 'Foundry Project Endpoint')
    search_endpoint = utils.get_deployment_output(output, 'searchServiceEndpoint', 'Search Service Endpoint')
    search_service_name = utils.get_deployment_output(output, 'searchServiceName', 'Search Service Name')
    ai_services_endpoint = utils.get_deployment_output(output, 'aiServicesEndpoint', 'AI Services Endpoint')
    ai_services_name = utils.get_deployment_output(output, 'aiServicesName', 'AI Services Name')
    ai_search_api_path = utils.get_deployment_output(output, 'searchAPIPath', 'Search API Path')

<a id='4'></a>
### 4️⃣ Create Search Index with Agentic Retrieval Support

Create an Azure AI Search index configured for Foundry IQ Agentic Retrieval with:
- **Vector search** — HNSW algorithm with Azure OpenAI vectorizer
- **Semantic search** — Semantic reranking for improved relevance
- **Auto-vectorization** — Vectorizer calls OpenAI embeddings at query time

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, VectorSearch, VectorSearchProfile,
    HnswAlgorithmConfiguration, AzureOpenAIVectorizer, AzureOpenAIVectorizerParameters,
    SemanticSearch, SemanticConfiguration, SemanticPrioritizedFields, SemanticField
)

credential = DefaultAzureCredential()
#### Direct access to Search Service using DefaultAzureCredential with Managed Identity (if running in Azure) or Azure CLI credentials (if running locally)
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)

# Create search index with vector + semantic search
index = SearchIndex(
    name=index_name,
    fields=[
        SearchField(name="id", type="Edm.String", key=True, filterable=True, sortable=True),
        SearchField(name="title", type="Edm.String", searchable=True, filterable=True),
        SearchField(name="content", type="Edm.String", searchable=True),
        SearchField(name="content_vector", type="Collection(Edm.Single)", stored=False,
                    vector_search_dimensions=3072, vector_search_profile_name="vector_profile"),
        SearchField(name="source", type="Edm.String", filterable=True),
        SearchField(name="page_number", type="Edm.Int32", filterable=True, sortable=True),
        SearchField(name="category", type="Edm.String", filterable=True, facetable=True)
    ],
    vector_search=VectorSearch(
        profiles=[VectorSearchProfile(name="vector_profile",
                                      algorithm_configuration_name="hnsw_config",
                                      vectorizer_name="azure_openai_vectorizer")],
        algorithms=[HnswAlgorithmConfiguration(name="hnsw_config")],
        vectorizers=[
            AzureOpenAIVectorizer(
                vectorizer_name="azure_openai_vectorizer",
                parameters=AzureOpenAIVectorizerParameters(
                    resource_url=ai_services_endpoint,
                    deployment_name=embedding_model,
                    model_name=embedding_model
                )
            )
        ]
    ),
    semantic_search=SemanticSearch(
        default_configuration_name="semantic_config",
        configurations=[
            SemanticConfiguration(
                name="semantic_config",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="title"),
                    content_fields=[SemanticField(field_name="content")]
                )
            )
        ]
    )
)

index_client.create_or_update_index(index)
utils.print_ok(f"Index '{index_name}' created successfully")

<a id='5'></a>
### 5️⃣ Upload sample documents

Generate embeddings via APIM gateway and upload documents to the search index. Note that embedding generation flows through the **APIM gateway**, demonstrating APIM-mediated traffic for AI operations.

In [ ]:
from azure.search.documents import SearchClient
from openai import AzureOpenAI

# Initialize OpenAI client pointing at APIM gateway to use the embedding model for vectorization with secure key management and unified observability
openai_client = AzureOpenAI(
    azure_endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
    api_key=api_key,
    api_version="2024-06-01"
)

# Sample documents about Azure AI services
sample_documents = [
    {
        "id": "1",
        "title": "Microsoft Agent Framework Overview",
        "content": "The Microsoft Agent Framework is an open-source, model-agnostic engine for building agentic AI applications. It provides unified agent abstractions, context providers for injecting relevant context, tool integration, and protocol support for A2A and AG-UI. The framework is available for both Python and .NET.",
        "source": "agent-framework-docs",
        "page_number": 1,
        "category": "framework"
    },
    {
        "id": "2",
        "title": "AzureAISearchContextProvider",
        "content": "The AzureAISearchContextProvider is a pluggable context provider in the Microsoft Agent Framework that connects agents to Azure AI Search. It supports two modes: semantic mode for fast hybrid search combining vector similarity, keyword matching, and semantic reranking; and agentic mode powered by Foundry IQ Knowledge Bases for intelligent multi-hop retrieval.",
        "source": "agent-framework-docs",
        "page_number": 2,
        "category": "context-providers"
    },
    {
        "id": "3",
        "title": "Foundry IQ Agentic Retrieval",
        "content": "Foundry IQ treats retrieval as a reasoning task, not just keyword lookup or vector search. It handles query planning where an LLM analyzes your query and plans optimal sub-queries, multi-hop reasoning that follows chains of information across documents, answer synthesis that returns comprehensive context with citations, and Knowledge Bases as a unified abstraction over multiple data sources.",
        "source": "foundry-iq-docs",
        "page_number": 3,
        "category": "retrieval"
    },
    {
        "id": "4",
        "title": "Retrieval Reasoning Effort Levels",
        "content": "Retrieval Reasoning Effort controls how much work the Foundry IQ engine does when planning queries. Low effort bypasses LLM query planning reducing cost and latency. Medium provides a balanced approach. High maximizes LLM processing for improved relevance on complex multi-hop queries. Microsoft evaluations show up to 36% improvement in response relevance on complex queries compared to traditional RAG.",
        "source": "foundry-iq-docs",
        "page_number": 4,
        "category": "configuration"
    },
    {
        "id": "5",
        "title": "Semantic vs Agentic Mode",
        "content": "Semantic mode provides fast hybrid search combining vector similarity, keyword matching, and semantic reranking. It is ideal for straightforward queries where speed matters. Agentic mode uses Foundry IQ Knowledge Bases for intelligent retrieval with query planning, multi-hop reasoning, and answer synthesis. It handles complex multi-step questions that traditional RAG struggles with.",
        "source": "foundry-iq-docs",
        "page_number": 5,
        "category": "modes"
    },
    {
        "id": "6",
        "title": "APIM as AI Gateway for Foundry IQ",
        "content": "Azure API Management can serve as an AI Gateway mediating all traffic between clients, Azure OpenAI, and Azure AI Search. APIM provides token rate limiting, semantic caching, load balancing across model deployments, and unified observability. When combined with Foundry IQ, APIM adds managed identity authentication, token metrics emission, and centralized access control to the knowledge retrieval pipeline.",
        "source": "gateway-docs",
        "page_number": 6,
        "category": "gateway"
    }
]

# Generate embeddings via APIM gateway
utils.print_info("Generating vector embeddings via APIM gateway...")
for doc in sample_documents:
    response = openai_client.embeddings.create(input=doc["content"], model=embedding_model)
    doc["content_vector"] = response.data[0].embedding
    utils.print_info(f"  ✓ {doc['title']}")

# Upload documents to search index
search_client = SearchClient(endpoint=search_endpoint, index_name=index_name, credential=credential)
result = search_client.upload_documents(documents=sample_documents)
succeeded = sum(1 for r in result if r.succeeded)
utils.print_ok(f"Uploaded {succeeded}/{len(sample_documents)} documents to index '{index_name}'")

<a id='6'></a>
### 6️⃣ Create Knowledge Source and Knowledge Base

Create the Foundry IQ primitives:
- **Knowledge Source** — References the search index and defines which fields to include in citations
- **Knowledge Base** — Configures the agentic retrieval pipeline with `EXTRACTIVE_DATA` output mode and `minimal` reasoning effort

In [ ]:
from azure.search.documents.indexes.models import (
    SearchIndexKnowledgeSource, SearchIndexKnowledgeSourceParameters, SearchIndexFieldReference,
    KnowledgeBase, KnowledgeSourceReference,
    KnowledgeRetrievalOutputMode, KnowledgeRetrievalMinimalReasoningEffort, KnowledgeBaseAzureOpenAIModel
)

# Create Knowledge Source
knowledge_source = SearchIndexKnowledgeSource(
    name=knowledge_source_name,
    description="Agent Framework and Foundry IQ documentation",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=index_name,
        source_data_fields=[
            SearchIndexFieldReference(name="id"),
            SearchIndexFieldReference(name="title"),
            SearchIndexFieldReference(name="source"),
            SearchIndexFieldReference(name="page_number"),
            SearchIndexFieldReference(name="category")
        ]
    )
)

index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
utils.print_ok(f"Knowledge Source '{knowledge_source_name}' created")

In [ ]:
# Configure Azure OpenAI connection parameters
# Uses proxied inferencing via APIM gateway, pointing the vectorizer to APIM instead of AI Services directly
aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=apim_resource_gateway_url,
    deployment_name=agent_model,
    model_name=agent_model,
    api_key=subscription_key
)

# Create Knowledge Base with EXTRACTIVE_DATA mode
knowledge_base = KnowledgeBase(
    name=knowledge_base_name,
    description="Knowledge base for Agent Framework with Foundry IQ and APIM Gateway documentation",
    knowledge_sources=[
        KnowledgeSourceReference(name=knowledge_source_name)
    ],
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
)

index_client.create_or_update_knowledge_base(knowledge_base=knowledge_base)
utils.print_ok(f"Knowledge Base '{knowledge_base_name}' created")

<a id='7'></a>
### 7️⃣ Create Agent with Microsoft Agent Framework (Agentic Mode - Knowledge Base)

Create a `ChatAgent` using the Microsoft Agent Framework with the `AzureAISearchContextProvider` in **agentic mode**.
This uses Foundry IQ Knowledge Bases for intelligent, multi-hop retrieval — the agent plans queries, follows chains of
information, and synthesizes answers across documents.

~20 lines of Python. Enterprise-grade RAG. Powered by Foundry IQ.

Reference: [Foundry IQ in Microsoft Agent Framework](https://devblogs.microsoft.com/foundry/foundry-iq-agent-framework-integration/)

In [ ]:
from agent_framework import Agent
from agent_framework.azure import AzureAISearchContextProvider, AzureOpenAIChatClient
from azure.identity.aio import DefaultAzureCredential as AsyncDefaultAzureCredential

agent_instructions = """You are a helpful assistant that answers questions using only the provided context.
Every answer must provide citations from the retrieved documents.
If you cannot find the answer in the provided context, respond with 'I don't know'.
"""

async def ask_agent_agentic(question: str):
    """Ask a question using Agent Framework with Foundry IQ agentic retrieval."""
    utils.print_info(f"Question: {question}")
    print("=" * 60)

    async_credential = AsyncDefaultAzureCredential()
    client = AzureOpenAIChatClient(
        endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
        deployment_name=agent_model,
        api_key=subscription_key
    )

    async with (
        AzureAISearchContextProvider(
            endpoint=f"{apim_resource_gateway_url}/{ai_search_api_path}",
            knowledge_base_name=knowledge_base_name,
            api_key=subscription_key,
            mode="agentic",
        ) as search,
        Agent(
            client=client,
            instructions=agent_instructions,
            context_providers=[search],
        ) as agent,
    ):
        response = await agent.run(question)
        utils.print_message(f"Answer (agentic):\n{response.text}")

    await async_credential.close()
    return response

utils.print_ok('Agent Framework (agentic mode) ready')

<a id='agentic'></a>
### 🧪 Test Agentic Mode (Foundry IQ)

The `AzureAISearchContextProvider` in agentic mode uses Foundry IQ Knowledge Bases for intelligent retrieval.
It plans queries, follows chains of information, and synthesizes answers — handling complex, multi-step questions
that traditional RAG struggles with.

Tip: Use the [tracing tool](../../tools/tracing.ipynb) to track the behavior and troubleshoot the [policy](policy.xml).

In [ ]:
# Test: Ask about the Agent Framework
response = await ask_agent_agentic("What is the Microsoft Agent Framework and what does it provide?")

In [ ]:
# Test: Ask about retrieval modes
response = await ask_agent_agentic("What is the difference between semantic and agentic retrieval modes?")

In [ ]:
# Test: Ask about APIM as AI Gateway
response = await ask_agent_agentic("What role does Azure API Management play in the Foundry IQ architecture?")

<a id='8'></a>
### 8️⃣ Create Agent with Semantic Mode (Index-based)

Create a `ChatAgent` using the `AzureAISearchContextProvider` in **semantic mode**.
This uses fast [hybrid search](https://learn.microsoft.com/azure/search/hybrid-search-overview) combining
vector similarity, keyword matching, and [semantic reranking](https://learn.microsoft.com/azure/search/semantic-search-overview).
Great for straightforward queries where speed matters.

In [ ]:
async def ask_agent_semantic(question: str):
    """Ask a question using Agent Framework with semantic (fast hybrid) retrieval."""
    utils.print_info(f"Question: {question}")
    print("=" * 60)

    async_credential = AsyncDefaultAzureCredential()
    client = AzureOpenAIChatClient(
        endpoint=f"{apim_resource_gateway_url}/{inference_api_path}",
        deployment_name=agent_model,
        api_key=subscription_key
    )

    async with (
        AzureAISearchContextProvider(
            endpoint=f"{apim_resource_gateway_url}/{ai_search_api_path}",
            api_key=subscription_key,
            index_name=index_name,
            mode="semantic",
            top_k=5,
        ) as search,
        Agent(
            client=client,
            instructions=agent_instructions,
            context_providers=[search],
        ) as agent,
    ):
        response = await agent.run(question)
        utils.print_message(f"Answer (semantic):\n{response.text}")

    await async_credential.close()
    return response

utils.print_ok('Agent Framework (semantic mode) ready')

<a id='semantic'></a>
### 🧪 Test Semantic Mode

Semantic mode is faster and simpler — great for straightforward queries where speed matters.

In [ ]:
# Test: Ask about Foundry IQ
response = await ask_agent_semantic("What is Foundry IQ and how does it improve retrieval?")

In [ ]:
# Test: Ask about retrieval reasoning effort
response = await ask_agent_semantic("What are the Retrieval Reasoning Effort levels?")

<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up-resources notebook](clean-up-resources.ipynb) for that.